# Notebook 0: data prep and shared evaluation code
**Run on Kaggle with Internet ON. No GPU needed.**

Produces `/kaggle/working/nli_data/` containing:
- `nli_common.py` : shared labels, metrics, `save_run()`, McNemar test (imported by every later notebook)
- `data/` : cleaned SNLI (train 100K ⊂ nested 50K, full val, full test) and MNLI (matched, mismatched) as CSVs
- `data/manifest.json` : counts and settings, for the report

After it finishes: **Save Version → Save & Run All**, then create a Kaggle Dataset named `nli-data` from this notebook's output.
Later notebooks attach `nli-data` as input and do `sys.path.append("/kaggle/input/nli-data")`.

In [1]:
import json, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
N_TRAIN = 100_000       # main training set for ALL models
N_SMALL = 50_000        # nested subset (for a data-scaling point / quick runs)
ROOT = Path("/kaggle/working/nli_data")
DATA = ROOT / "data"
DATA.mkdir(parents=True, exist_ok=True)

## 1. Shared module (single source of truth for metrics and file formats)

In [2]:
%%writefile /kaggle/working/nli_data/nli_common.py
"""Shared code for all NLI notebooks (LR, DistilBERT, Llama-LoRA, prompted Llama, novelty).

Every model, on every evaluation set, goes through save_run() so that prediction files and
result JSONs have exactly the same format and always record the config that actually ran.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

SEED = 42
LABELS = ["entailment", "neutral", "contradiction"]      # 0, 1, 2 (same for SNLI and MNLI)
PROB_COLS = ["p_ent", "p_neu", "p_con"]
DATA_DIR_DEFAULT = "/kaggle/input/nli-data/data"         # where Notebook 0's output is mounted

# Evaluation sets. MNLI test labels are hidden, so the two validation sets are used (never trained/tuned on).
EVAL_SETS = ["snli_val", "snli_test", "mnli_m", "mnli_mm"]
FILES = {
    "train_100k": "snli_train_100k.csv",
    "train_50k": "snli_train_50k.csv",
    "snli_val": "snli_val.csv",
    "snli_test": "snli_test.csv",
    "mnli_m": "mnli_val_matched.csv",
    "mnli_mm": "mnli_val_mismatched.csv",
}


def load_split(name, data_dir=DATA_DIR_DEFAULT):
    """Load one split as a DataFrame with columns id, premise, hypothesis, label (+genre for MNLI)."""
    df = pd.read_csv(Path(data_dir) / FILES[name], keep_default_na=False)
    df["premise"] = df["premise"].astype(str)
    df["hypothesis"] = df["hypothesis"].astype(str)
    df["label"] = df["label"].astype(int)
    assert df["label"].isin([0, 1, 2]).all(), f"{name}: labels outside {{0,1,2}}"
    return df


def evaluate(y_true, pred):
    y_true, pred = np.asarray(y_true), np.asarray(pred)
    p, r, f, s = precision_recall_fscore_support(y_true, pred, labels=[0, 1, 2], zero_division=0)
    return {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "f1_macro": float(f1_score(y_true, pred, average="macro")),
        "f1_weighted": float(f1_score(y_true, pred, average="weighted")),
        "per_class": {l: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])}
                      for i, l in enumerate(LABELS)},
        "confusion_matrix": confusion_matrix(y_true, pred, labels=[0, 1, 2]).tolist(),   # rows=gold, cols=pred
    }


def per_genre_accuracy(df, pred):
    d = df.assign(correct=(df["label"].to_numpy() == np.asarray(pred)))
    return {g: float(v) for g, v in d.groupby("genre")["correct"].mean().items()}


def save_run(out_dir, model, eval_set, df, probs, config, extra=None):
    """Save predictions CSV + results JSON for one model on one eval set.

    model     : name incl. seed, e.g. 'distilbert_s42', 'lr', 'llama1b_lora', 'llama1b_fewshot6'
    eval_set  : one of EVAL_SETS
    df        : the evaluation DataFrame (id, label, optional genre) in the order probs were computed
    probs     : (N,3) array of class probabilities in label order [ent, neu, con]
    config    : dict of the exact hyperparameters / settings that ran (stored in the JSON)
    extra     : optional dict (train_time_s, inference_s, n_params, gpu_mem_gb, ...)
    """
    assert eval_set in EVAL_SETS, eval_set
    probs = np.asarray(probs, dtype=np.float64)
    assert probs.shape == (len(df), 3), f"probs shape {probs.shape} vs {len(df)} rows"
    pred = probs.argmax(1)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cols = ["id", "label"] + (["genre"] if "genre" in df.columns else [])
    pdf = df[cols].copy().reset_index(drop=True)
    pdf["pred"] = pred
    for i, c in enumerate(PROB_COLS):
        pdf[c] = probs[:, i]
    pdf.to_csv(out_dir / f"preds_{model}_{eval_set}.csv", index=False)

    res = {"model": model, "eval_set": eval_set, "config": config, "extra": extra or {},
           "metrics": evaluate(df["label"].to_numpy(), pred)}
    if "genre" in df.columns:
        res["metrics"]["per_genre_accuracy"] = per_genre_accuracy(df, pred)
    with open(out_dir / f"results_{model}_{eval_set}.json", "w") as f:
        json.dump(res, f, indent=2, default=str)
    m = res["metrics"]
    print(f"[{model} | {eval_set}] acc={m['accuracy']:.4f} macroF1={m['f1_macro']:.4f} "
          f"per-class F1={[round(m['per_class'][l]['f1'], 3) for l in LABELS]}")
    return res


def load_preds(out_dir, model, eval_set):
    return pd.read_csv(Path(out_dir) / f"preds_{model}_{eval_set}.csv")


def mcnemar(pred_a, pred_b, y_true):
    """Exact McNemar test on paired predictions. Returns (b, c, p): b = only A right, c = only B right."""
    from scipy.stats import binomtest
    a_ok, b_ok = np.asarray(pred_a) == np.asarray(y_true), np.asarray(pred_b) == np.asarray(y_true)
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    p = 1.0 if b + c == 0 else float(binomtest(min(b, c), b + c, 0.5).pvalue)
    return b, c, p

Writing /kaggle/working/nli_data/nli_common.py


In [3]:
sys.path.insert(0, str(ROOT))
import importlib, nli_common
importlib.reload(nli_common)
from nli_common import LABELS, FILES, SEED as _S
assert _S == SEED

## 2. SNLI: load, drop invalid labels (-1), subset

In [4]:
from datasets import load_dataset

def clean(df, name):
    n0 = len(df)
    df = df[df["label"].isin([0, 1, 2])].copy()
    n_invalid = n0 - len(df)
    for c in ("premise", "hypothesis"):
        df[c] = df[c].fillna("").astype(str).str.strip()
    empty = ((df["premise"] == "") | (df["hypothesis"] == "")).sum()
    df = df[(df["premise"] != "") & (df["hypothesis"] != "")].reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    df.insert(0, "id", np.arange(len(df)))        # stable id = row index in the cleaned split
    print(f"{name:<12} {n0:>7,} -> {len(df):>7,} rows  (invalid labels removed: {n_invalid}, empty text removed: {empty})")
    return df

snli = load_dataset("stanfordnlp/snli")
tr_full = clean(snli["train"].to_pandas(), "snli_train")
val = clean(snli["validation"].to_pandas(), "snli_val")
test = clean(snli["test"].to_pandas(), "snli_test")
# Expected after cleaning: 549,367 / 9,842 / 9,824 (matches the Phase 2 report)

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  412kB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  413kB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

snli_train   550,152 -> 549,367 rows  (invalid labels removed: 785, empty text removed: 0)
snli_val      10,000 ->   9,842 rows  (invalid labels removed: 158, empty text removed: 0)
snli_test     10,000 ->   9,824 rows  (invalid labels removed: 176, empty text removed: 0)


In [5]:
# Nested, stratified subsets: 50K ⊂ 100K ⊂ full. Fixed seed, so every notebook sees identical rows.
tr100, _ = train_test_split(tr_full, train_size=N_TRAIN, stratify=tr_full["label"], random_state=SEED)
tr50, _ = train_test_split(tr100, train_size=N_SMALL, stratify=tr100["label"], random_state=SEED)
tr100, tr50 = tr100.reset_index(drop=True), tr50.reset_index(drop=True)     # 'id' column keeps the full-train row id
assert set(tr50["id"]) <= set(tr100["id"])
print("100K class counts:", tr100["label"].value_counts().sort_index().to_dict())
print(" 50K class counts:", tr50["label"].value_counts().sort_index().to_dict())

100K class counts: {0: 33387, 1: 33268, 2: 33345}
 50K class counts: {0: 16693, 1: 16634, 2: 16673}


## 3. MNLI (evaluation only, never trained or tuned on)

In [6]:
mnli = load_dataset("nyu-mll/multi_nli")
print(mnli)

def clean_mnli(df, name):
    d = clean(df[["premise", "hypothesis", "label", "genre"]].copy(), name)
    return d

mnli_m = clean_mnli(mnli["validation_matched"].to_pandas(), "mnli_m")
mnli_mm = clean_mnli(mnli["validation_mismatched"].to_pandas(), "mnli_mm")
print("matched genres   :", mnli_m["genre"].value_counts().to_dict())
print("mismatched genres:", mnli_mm["genre"].value_counts().to_dict())

README.md:   0%|          | 0.00/8.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  214MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation_matched-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 4.94MB            

data/validation_matched-00000-of-00001.p(…): downloading bytes:           |  0.00B            

data/validation_mismatched-00000-of-0000(…): reconstructing file:   0%|          |  0.00B / 5.10MB            

data/validation_mismatched-00000-of-0000(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 9832
    })
})
mnli_m         9,815 ->   9,815 rows  (invalid labels removed: 0, empty text removed: 0)
mnli_mm        9,832 ->   9,832 rows  (invalid labels removed: 0, empty text removed: 0)
matched genres   : {'travel': 1976, 'fiction': 1973, 'telephone': 1966, 'slate': 1955, 'government': 1

## 4. Sanity checks (these numbers go into the Dataset section of the report)

In [7]:
def pairs(df): return set(zip(df["premise"], df["hypothesis"]))
def prem(df): return set(df["premise"])

stats = {}
print("Exact (premise, hypothesis) overlap of train_100k with:")
for n, d in [("snli_val", val), ("snli_test", test), ("mnli_m", mnli_m), ("mnli_mm", mnli_mm)]:
    stats[f"pair_overlap_{n}"] = len(pairs(tr100) & pairs(d))
    stats[f"premise_overlap_{n}"] = len(prem(tr100) & prem(d))
    print(f"  {n:<9} pairs={stats[f'pair_overlap_{n}']:<4} premises={stats[f'premise_overlap_{n}']}")

wc = lambda s: s.str.split().str.len()
len_stats = {}
for n, d in [("snli_train_100k", tr100), ("snli_test", test), ("mnli_m", mnli_m), ("mnli_mm", mnli_mm)]:
    p, h = wc(d["premise"]), wc(d["hypothesis"])
    len_stats[n] = {"premise_mean": float(p.mean()), "premise_p95": float(p.quantile(.95)), "premise_max": int(p.max()),
                    "hyp_mean": float(h.mean()), "hyp_p95": float(h.quantile(.95)), "hyp_max": int(h.max())}
    print(f"{n:<16} premise words mean={p.mean():5.1f} p95={p.quantile(.95):5.0f} max={p.max():4d} | "
          f"hyp mean={h.mean():5.1f} p95={h.quantile(.95):4.0f} max={h.max():4d}")
print("-> MNLI premises are much longer than SNLI; use max_len >= 256 for transformer inference on MNLI.")

Exact (premise, hypothesis) overlap of train_100k with:
  snli_val  pairs=0    premises=7
  snli_test pairs=0    premises=6
  mnli_m    pairs=0    premises=0
  mnli_mm   pairs=0    premises=0
snli_train_100k  premise words mean= 12.8 p95=   23 max=  78 | hyp mean=  7.4 p95=  13 max=  46
snli_test        premise words mean= 13.9 p95=   25 max=  54 | hyp mean=  7.5 p95=  13 max=  29
mnli_m           premise words mean= 19.3 p95=   43 max= 196 | hyp mean=  9.9 p95=  17 max=  46
mnli_mm          premise words mean= 19.6 p95=   42 max= 129 | hyp mean= 10.8 p95=  19 max=  53
-> MNLI premises are much longer than SNLI; use max_len >= 256 for transformer inference on MNLI.


## 5. Save

In [8]:
cols = ["id", "premise", "hypothesis", "label"]
tr100[cols].to_csv(DATA / FILES["train_100k"], index=False)
tr50[cols].to_csv(DATA / FILES["train_50k"], index=False)
val[cols].to_csv(DATA / FILES["snli_val"], index=False)
test[cols].to_csv(DATA / FILES["snli_test"], index=False)
mnli_m[cols + ["genre"]].to_csv(DATA / FILES["mnli_m"], index=False)
mnli_mm[cols + ["genre"]].to_csv(DATA / FILES["mnli_mm"], index=False)

manifest = {
    "seed": SEED, "n_train": N_TRAIN, "n_small": N_SMALL, "labels": LABELS,
    "rows": {"snli_train_full_clean": len(tr_full), "snli_train_100k": len(tr100), "snli_train_50k": len(tr50),
             "snli_val": len(val), "snli_test": len(test), "mnli_m": len(mnli_m), "mnli_mm": len(mnli_mm)},
    "class_counts_100k": {LABELS[k]: int(v) for k, v in tr100["label"].value_counts().sort_index().items()},
    "overlap": stats, "length_stats_words": len_stats,
}
with open(DATA / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest["rows"], indent=2))

{
  "snli_train_full_clean": 549367,
  "snli_train_100k": 100000,
  "snli_train_50k": 50000,
  "snli_val": 9842,
  "snli_test": 9824,
  "mnli_m": 9815,
  "mnli_mm": 9832
}


## 6. Smoke test of the shared evaluation code (random probabilities, must run without errors)

In [9]:
from nli_common import load_split, save_run, load_preds, mcnemar
d = load_split("snli_val", str(DATA))
rng = np.random.default_rng(0)
probs = rng.dirichlet(np.ones(3), size=len(d))
res = save_run("/kaggle/working/_smoke", "random", "snli_val", d, probs, {"note": "smoke test"})
dm = load_split("mnli_m", str(DATA))
save_run("/kaggle/working/_smoke", "random", "mnli_m", dm, rng.dirichlet(np.ones(3), size=len(dm)), {"note": "smoke test"})
print(load_preds("/kaggle/working/_smoke", "random", "mnli_m").head(3))
print("McNemar (random vs random):", mcnemar(probs.argmax(1), rng.integers(0, 3, len(d)), d["label"]))
assert abs(res["metrics"]["accuracy"] - 1/3) < 0.03, "random baseline should be ~33%"
import shutil; shutil.rmtree("/kaggle/working/_smoke")
print("\nAll good. Now: Save Version -> Save & Run All, then create Dataset 'nli-data' from this notebook's output.")

[random | snli_val] acc=0.3314 macroF1=0.3314 per-class F1=[0.331, 0.329, 0.334]
[random | mnli_m] acc=0.3309 macroF1=0.3307 per-class F1=[0.338, 0.33, 0.324]
   id  label       genre  pred     p_ent     p_neu     p_con
0   0      1       slate     1  0.338639  0.654341  0.007019
1   1      2  government     1  0.024025  0.508210  0.467765
2   2      0   telephone     2  0.001804  0.047589  0.950607
McNemar (random vs random): (2164, 2184, 0.7732407702756902)

All good. Now: Save Version -> Save & Run All, then create Dataset 'nli-data' from this notebook's output.
